In [1]:
import sys
import yaml
from pathlib import Path

import numpy as np 
import torch 

from eb_jepa.datasets.utils import init_data
from eb_jepa.training_utils import load_config
from eb_jepa.datasets.two_rooms.env import DotWall
from eb_jepa.vis_utils import create_comparison_gif, show_images

PKG_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "cfgs" / "train.yaml").exists()
)
sys.path.insert(0, str(PKG_ROOT))

from builders import build_model
from planning import GCAgent


TRAIN_CFG_PATH = PKG_ROOT / "cfgs" / "train.yaml"
EVAL_CFG_PATH  = PKG_ROOT / "cfgs" / "eval.yaml"

### Train cfg

In [2]:
cfg = load_config(TRAIN_CFG_PATH)  # in order to use dot notation 

loader, val_loader, data_config = init_data(
    env_name=cfg.data.env_name,cfg_data=dict(cfg.data)
)

[INFO    ][2026-09-01 07:20:58][eb_jepa.training_utils][load_config              ] Loaded config from /Users/hawardizayee/Desktop/AMI/eb_jepa/examples/my_ac_video_jepa/cfgs/train.yaml


/Users/hawardizayee/Desktop/AMI/eb_jepa/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 10 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


### Eval cfg

In [3]:
with open(EVAL_CFG_PATH, "r") as f:
    eval_cfg_dict = yaml.safe_load(f)

# loader, val_loader, env_config
_, _, env_config = init_data(
    env_name="two_rooms",
    cfg_data=dict(eval_cfg_dict.get("data",{}))
)

cfg_eval_env = eval_cfg_dict.get("env")

def env_creator():
    cfg_eval_env = eval_cfg_dict.get("env")
    return DotWall(
        config=env_config,
        **cfg_eval_env
    )

### JEPA

In [4]:
jepa, xy_prober = build_model(
    cfg, 
    data_config=data_config, 
    normalizer=loader.dataset.normalizer
    )

# `main_unroll_eval`

In [5]:
env = env_creator()
env.reset()
# ========= Agent ========= 
agent = GCAgent(
    model = jepa,
    plan_cfg=None,
    normalizer=val_loader.dataset.normalizer,
    env = env, 
    loc_prober= xy_prober 
)
num_samples = 4

mse_values = []
position_mse_values = []
unroll = times = []
val_loader_iter = iter(val_loader)

for idx in range(num_samples):
    val_loader_batch = next(val_loader_iter)        # Batch Size 4 
    x, a, loc, wall_x, door_y = val_loader_batch

    with torch.no_grad(): 
        obs_init = x[:, :, 0:1]
        print(f'************unrolling for predicted_states************')
        predicted_states = agent.unroll(
            obs_init,
            a,
            repeat_batch=False 
        ) [:, :, :-1]  # throwing away the last prediction
        # (B, f=512, nsteps+1=18, 1, 1) throwing the last one (B,D=512, T=17, 1,1)
        

        print(f'************unrolling for random predicted_states************')
        rand_predicted_states = agent.unroll(
            obs_init,
            torch.randn_like(a),
            repeat_batch=False
        )[:, :, :-1]    # throwing away the last prediction

        B = 4 
        
        # shape manipulation before passing to encoding, is for bachifying. 
        gt_encoded = (
            jepa.encode(x.permute(0, 2, 1, 3, 4).flatten(0, 1).unsqueeze(2))
            .squeeze(2)
            .unflatten(dim=0, sizes=(B, -1))
            .permute(0,2,1,3,4)
            )
        

        latent_mse = (((gt_encoded - predicted_states)**2).mean(dim = (1, 3, 4)))   # (B=4, T=17)
        mse_values.append(latent_mse)

        if xy_prober:
            # ========= recover dot from representation =========
            gt_decoded = agent.decode_loc_to_pixel(gt_encoded, wall_x, door_y)
            pred_decoded = agent.decode_loc_to_pixel(predicted_states, wall_x, door_y)
            rand_pred_decoded = agent.decode_loc_to_pixel(rand_predicted_states, wall_x, door_y)

            gt_frames = agent.normalizer.unnormalize_state(x.permute(0, 2, 1, 3, 4)).permute(0, 1, 3, 4, 2)
            gt_frames = (gt_frames * 255).clamp(0, 255).to(torch.uint8).numpy()
            # print(f'gt_frames : {gt_frames.shape}')

            pred_positions = xy_prober.apply_head(predicted_states).permute(0, 2, 1)
            # print(f"pred positions : {pred_positions.shape}")

        gt_positions = loc.permute(0, 2, 1) # B T 2 

        position_mse = (
            ((pred_positions - gt_positions.cpu()) ** 2)
            .mean(dim=-1)
            .cpu()
            .numpy()
        )
        position_mse_values.append(position_mse)

        # create_comparison_gif(
        #     gt_frames, 
        #     pred_decoded,
        #     rand_pred_decoded,
        #     gt_dec=gt_decoded,
        #     save_path=f"1.gif"
        # )

[INFO    ][2026-09-01 07:20:58][planning            ][__init__                 ] No plan_cfg provided in GCAgent, planner not initialized.
************unrolling for predicted_states************
context_state : torch.Size([4, 512, 1, 1, 1])
slice [0 : 1]
context_actions : torch.Size([4, 2, 1])
torch.Size([4, 512, 1, 1, 1])
pred_step : torch.Size([4, 512, 1, 1, 1])
predicted_states cat : torch.Size([4, 512, 2, 1, 1])
context_state : torch.Size([4, 512, 1, 1, 1])
slice [1 : 2]
context_actions : torch.Size([4, 2, 1])
torch.Size([4, 512, 1, 1, 1])
pred_step : torch.Size([4, 512, 1, 1, 1])
predicted_states cat : torch.Size([4, 512, 3, 1, 1])
context_state : torch.Size([4, 512, 1, 1, 1])
slice [2 : 3]
context_actions : torch.Size([4, 2, 1])
torch.Size([4, 512, 1, 1, 1])
pred_step : torch.Size([4, 512, 1, 1, 1])
predicted_states cat : torch.Size([4, 512, 4, 1, 1])
context_state : torch.Size([4, 512, 1, 1, 1])
slice [3 : 4]
context_actions : torch.Size([4, 2, 1])
torch.Size([4, 512, 1, 1, 1])
p

In [6]:
results = {}

# latent mse metrics 

In [7]:
all_mse_values = np.vstack(mse_values)  # Shape: [num_batches, T]
mean_mse_per_timestep = np.mean(all_mse_values, axis=0) # shape: [T]
std_mse_timestep = np.std(all_mse_values, axis=0) # shape: [T]

for t in range(mean_mse_per_timestep.shape[0]):
    results[f"val_rollout/mean_mse/{t}"] = mean_mse_per_timestep[t]
    results[f"val_rollout/std_mse/{t}"] = std_mse_timestep[t]

# Position mse metrics 

In [8]:
all_position_mse_values = np.vstack(position_mse_values)    # Shape: [num_batches, T]
mean_position_mse_per_timestep = np.mean(all_position_mse_values, axis=0) # [T]
std_position_mse_per_timestep = np.std(all_position_mse_values, axis=0)

for t in range(mean_position_mse_per_timestep.shape[0]):
    results[f"val_rollout/mean_pos_mse/{t}"] = mean_position_mse_per_timestep[t]
    results[f"val_rollout/std_pos_mse/{t}"] = std_position_mse_per_timestep[t]

In [9]:

import pandas as pd 

pd.DataFrame([results])


,val_rollout/mean_mse/0,val_rollout/std_mse/0,val_rollout/mean_mse/1,val_rollout/std_mse/1,val_rollout/mean_mse/2,val_rollout/std_mse/2,val_rollout/mean_mse/3,val_rollout/std_mse/3,val_rollout/mean_mse/4,val_rollout/std_mse/4,...,val_rollout/mean_pos_mse/12,val_rollout/std_pos_mse/12,val_rollout/mean_pos_mse/13,val_rollout/std_pos_mse/13,val_rollout/mean_pos_mse/14,val_rollout/std_pos_mse/14,val_rollout/mean_pos_mse/15,val_rollout/std_pos_mse/15,val_rollout/mean_pos_mse/16,val_rollout/std_pos_mse/16
0,2.174456e-12,5.289606e-13,0.15394,0.015666,0.406213,0.030095,0.65162,0.042042,0.869959,0.049299,...,1.376465,0.650269,1.35658,0.637762,1.333475,0.620359,1.306853,0.631224,1.274744,0.634928
